#Start

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive/')
cwd="/content/drive/My Drive/ACL-Parsing-2026" #the working directory on Google drive is ACL-Parsing-2026, you can change it to any directory name
if not os.path.exists(cwd): os.makedirs(cwd)
os.chdir(cwd)

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


#Load Gitlab Data

In [ ]:
#first time only
#!rm -r parsing_utils #in case we need to clone the repo after update
!git clone https://gitlab.com/acl2575601/acl-parsing-2026.git parsing_utils

In [5]:
#updating Repo
os.chdir("parsing_utils")
!git pull
os.chdir(cwd)

Already up to date.


#Load Rules from Spreadsheet
Loading syntactic rules from excel file in "rules" directory - run this cell every time you update something in the excel file for the rules sheet or word features sheet

In [22]:
#This cell converts the xlsx file with rules/word features into JSON file. You don't need to run it unless you change somehting in the xlsx file
#rule file is in directory parsing_utils/rules

from parsing_utils.code.pandas_utils import *
from itertools import groupby
import json

rules_dir="parsing_utils/rules"
rules_xls_fname="parsing_rules_en.xlsx"
rules_workbook_fpath=os.path.join(rules_dir,rules_xls_fname)

rules_json_fname=rules_xls_fname.replace(".xlsx",".json")
rules_json_fpath=os.path.join(rules_dir,rules_json_fname)

#rules_workbook_url="https://docs.google.com/spreadsheets/d/e/2PACX-1vQWwi-tdS4z1oXN3fELGqBSh6elDa2OtTIh7zFLZfufOON2PYIg2QBd9avPdXz2g3CwpmCj-tNV76Qn/pub?output=xlsx"

rw_data_dict=get_wb_data_dict(rules_workbook_fpath)

rw_data_dict.keys()
#rules_sheet=rw_data_dict["short-list"]
rules_sheet=rw_data_dict["rule-list-new"]

rules_sheet_v0=rw_data_dict["rules_v0"]

word_features_list=rw_data_dict["words-features"]
upos_sheet_rows=rw_data_dict["upos"]
xpos_sheet_rows=rw_data_dict["xpos"]

final_rules=[]
for r0 in rules_sheet:
  cur_rule=r0["Rule"].strip()
  if not cur_rule: continue
  final_rules.append(cur_rule)

final_rules_v0=[]
for r0 in rules_sheet_v0:
  cur_rule=r0["Rule"].strip()
  if not cur_rule: continue
  final_rules_v0.append(cur_rule)


upos_ft_dict={}
xpos_ft_dict={}
for a in  upos_sheet_rows:
  tag0,ft0=a["tag"],a["features"]
  upos_ft_dict[tag0]=ft0.strip().split()
upos_list=sorted(list(upos_ft_dict.keys()))

for a in  xpos_sheet_rows:
  #print(a)
  tag0,ft0,freq0=a["tag"],a["features"],int(a["freq"])
  if "-" in tag0 and not tag0.startswith("-"): continue
  if freq0<10: continue
  xpos_ft_dict[tag0]=ft0.strip().split()
xpos_list=sorted(list(xpos_ft_dict.keys()))

#print("upos_list",len(upos_list), upos_list)
#print("xpos_list",len(xpos_list), xpos_list)
word_features_list.sort(key=lambda x:x["word"])
for a in word_features_list:
  if a["category"]=="": continue
word_features_list_grouped=[(key,[(v["category"],v["features"].strip().split()) for v in list(group)]) for key,group in groupby(word_features_list,lambda x:x["word"])]
final_word_features_dict=dict(iter(word_features_list_grouped))


final_parsing_data_dict={}
final_parsing_data_dict["final_word_features_dict"]=final_word_features_dict
final_parsing_data_dict["upos_list"]=upos_list
final_parsing_data_dict["xpos_list"]=xpos_list
final_parsing_data_dict["xpos_ft_dict"]=xpos_ft_dict
final_parsing_data_dict["upos_ft_dict"]=upos_ft_dict
final_parsing_data_dict["final_rules"]=final_rules
final_parsing_data_dict["final_rules_v0"]=final_rules_v0

with open(rules_json_fpath,"w") as json_open:
  json.dump(final_parsing_data_dict,json_open)

print("finished processing rules and word features - saved into:",rules_json_fpath)


finished processing rules and word features - saved into: parsing_utils/rules/parsing_rules_en.json


#Apply Parser

In [23]:
from parsing_utils.code.parsing_lib import *
import json, re

json_fpath="parsing_utils/rules/parsing_rules_en.json"
with open(json_fpath) as json_fopen:
  final_parsing_data_dict=json.load(json_fopen)

rules=final_parsing_data_dict["final_rules"]

sent = "the man and the woman in the kitchen sing a song"

sent=re.sub(r"(\W)",r" \1 ",sent)
sent_toks=sent.strip().split()

rnn_model_fpath="parsing_utils/models/H64_L3_LR001_updated_xpos_loading.model"
#rnn_model_fpath=None
params0={"pos_model_path":rnn_model_fpath,"final_word_features_dict":final_word_features_dict,"xpos_ft_dict":xpos_ft_dict}
params0["debug"]=False
params0["max_n_phrases"]=3
params0["max_skip_distance"]=2
import time
#parser_obj=Parser(rules_list=rules,params=params0)
parser_obj=Parser(rules_list=rules,params=params0)
t0=time.time()
final_parses=parser_obj.parse(sent_toks)
t1=time.time()

from IPython.display import HTML, display
fp0,dep0,const0=final_parses[0]
html_output=viz_parse(dep0,const0)
display(HTML(html_output))
